# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connected.")

Warehouse connected.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check 1 — Staleness

I check whether content age is associated with the March observed outcome. This signal is linked to FlyRank's refresh/staleness logic. I use the March outcome only to audit whether the signal is informative; March is not used as an input to the final baseline score.

In [4]:
# Signal check 1 — staleness/content age
# Compare content-age buckets with the March observed outcome.

staleness_check = con.sql(f"""
WITH feb_age AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days
    FROM {FEB} f
    JOIN read_parquet('{DIM_CONTENT}') c
      ON f.client_hash_id = c.client_hash_id
     AND f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.content_created_date IS NOT NULL
    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        c.content_created_date
),

march_outcome AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS went_dark
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN a.content_age_days < 180 THEN 'under_180_days'
        WHEN a.content_age_days < 365 THEN '180_to_364_days'
        ELSE '365_plus_days'
    END AS age_bucket,
    COUNT(*) AS n,
    ROUND(AVG(m.went_dark), 4) AS march_went_dark_rate
FROM feb_age a
JOIN march_outcome m
  ON a.client_hash_id = m.client_hash_id
 AND a.content_hash_id = m.content_hash_id
GROUP BY 1
ORDER BY 1
""").df()

staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,march_went_dark_rate
0,180_to_364_days,56902,0.5706
1,365_plus_days,11199,0.5767
2,under_180_days,66137,0.5651


In [3]:
# Signal check 1 — content age buckets

# Inspect dim_content first so we use the real warehouse fields.
DIM_CONTENT = f"{REL}/dim_content.parquet"

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{DIM_CONTENT}')
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


### Signal check 2 — search visibility

I check whether February search impressions are associated with the March observed outcome. Impressions are linked to the volume/visibility logic used in quick-win prioritisation. This audit uses March only to test the signal; March information will not be used by the final baseline rule.

In [6]:
# Signal check 2 — February search-impression buckets

impression_check = con.sql(f"""
WITH feb_impressions AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march_outcome AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS went_dark
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

bucketed AS (
    SELECT
        CASE
            WHEN i.impressions < 100 THEN 'under_100'
            WHEN i.impressions < 1000 THEN '100_to_999'
            ELSE '1000_plus'
        END AS impression_bucket,
        m.went_dark
    FROM feb_impressions i
    JOIN march_outcome m
      ON i.client_hash_id = m.client_hash_id
     AND i.content_hash_id = m.content_hash_id
)

SELECT
    impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(went_dark), 4) AS march_went_dark_rate
FROM bucketed
GROUP BY impression_bucket
ORDER BY
    CASE
        WHEN impression_bucket = 'under_100' THEN 1
        WHEN impression_bucket = '100_to_999' THEN 2
        ELSE 3
    END
""").df()

impression_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,march_went_dark_rate
0,under_100,57536,0.8726
1,100_to_999,43794,0.5331
2,1000_plus,32908,0.0833


**Verdict: CONFIRMED**

The observed March `went_dark` rate decreases sharply as February search impressions increase: 0.8726 for pages with under 100 impressions, 0.5331 for 100–999 impressions, and 0.0833 for 1000+ impressions. This supports using search visibility as a baseline signal, while the relationship should be treated as directional rather than causal.

### Baseline scoring rule

I score each client × content item using only February 2026 decision-time information.

- Add **2 points** when content age is 365+ days.
- Add **1 point** when content age is 180–364 days.
- Add **2 points** when February GSC impressions are below 100.
- Add **1 point** when February GSC impressions are between 100 and 999.
- Score 0 otherwise.

The final score ranges from 0 to 4. Higher scores indicate a stronger directional review signal.

Reason codes:
- `stale_low_visibility` — old content with low search visibility; action: `review_refresh`
- `aging_low_visibility` — moderately old content with low search visibility; action: `review_refresh`
- `low_visibility` — low search visibility without the stronger age signal; action: `review_visibility`
- `stale` — old content without low visibility; action: `review_refresh`
- `monitor` — none of the review conditions are met; action: `monitor`

The rule uses only February information. It does not use the March outcome, `went_dark`, or any product-generated priority/health/action flag.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Build ranked baseline queue

import os
import pandas as pd

queue = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
),

scored AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions,
        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days,

        CASE
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-02-28') >= 365
                 AND f.gsc_impressions < 100
                THEN 4
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-02-28') >= 365
                 AND f.gsc_impressions < 1000
                THEN 3
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-02-28') BETWEEN 180 AND 364
                 AND f.gsc_impressions < 100
                THEN 3
            WHEN DATE_DIFF('day', c.content_created_date, DATE '2026-02-28') BETWEEN 180 AND 364
                 AND f.gsc_impressions < 1000
                THEN 2
            WHEN f.gsc_impressions < 100
                THEN 2
            WHEN f.gsc_impressions < 1000
                THEN 1
            ELSE 0
        END AS score

    FROM feb f
    JOIN content c
      ON f.client_hash_id = c.client_hash_id
     AND f.content_hash_id = c.content_hash_id
    WHERE c.content_created_date IS NOT NULL
)

SELECT
    client_hash_id,
    content_hash_id,
    content_age_days,
    gsc_impressions,
    score,

    CASE
        WHEN score >= 3 THEN 'stale_low_visibility'
        WHEN score = 2 AND content_age_days >= 180 THEN 'aging_low_visibility'
        WHEN score = 2 THEN 'low_visibility'
        WHEN score = 1 THEN 'visible_low_priority'
        ELSE 'monitor'
    END AS reason_code,

    CASE
        WHEN score >= 3 THEN 'review_refresh'
        WHEN score = 2 THEN 'review_refresh'
        WHEN score = 1 THEN 'review_visibility'
        ELSE 'monitor'
    END AS action

FROM scored
ORDER BY score DESC, content_age_days DESC, gsc_impressions ASC
""").df()

# Add rank
queue.insert(0, "rank", range(1, len(queue) + 1))

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write required CSV
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Queue rows:", len(queue))
print("CSV written to:", output_path)
print()
display(queue.head(10))


Queue rows: 153559
CSV written to: work/outputs/baseline_action_score.csv



,rank,client_hash_id,content_hash_id,content_age_days,gsc_impressions,score,reason_code,action
0,1,client_c182d11e4862a37d,content_a015022fdf20343b,463,5.0,4,stale_low_visibility,review_refresh
1,2,client_c182d11e4862a37d,content_af7f2771f9386f92,463,29.0,4,stale_low_visibility,review_refresh
2,3,client_c182d11e4862a37d,content_1003d0e15bd1a910,463,46.0,4,stale_low_visibility,review_refresh
3,4,client_c182d11e4862a37d,content_686064cfde37e2b3,463,84.0,4,stale_low_visibility,review_refresh
4,5,client_c182d11e4862a37d,content_455ee259df594322,456,1.0,4,stale_low_visibility,review_refresh
5,6,client_c182d11e4862a37d,content_abaf64d1ed51568f,456,1.0,4,stale_low_visibility,review_refresh
6,7,client_c182d11e4862a37d,content_69fa27a38394df56,456,1.0,4,stale_low_visibility,review_refresh
7,8,client_c182d11e4862a37d,content_9f3f62757ed3dfa9,456,1.0,4,stale_low_visibility,review_refresh
8,9,client_c182d11e4862a37d,content_b64e454944d3340f,456,1.0,4,stale_low_visibility,review_refresh
9,10,client_c182d11e4862a37d,content_5917bce4a37cd949,456,1.0,4,stale_low_visibility,review_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda r: (
        "Higher-priority baseline pick because the page is both older "
        "and has low search visibility."
        if r["score"] >= 3
        else
        "Moderate baseline pick based on the observed age/visibility signals."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The page may be intentionally low-visibility, seasonal, "
    "already scheduled for change, or its low impressions may reflect "
    "limited search demand rather than a refresh opportunity."
)

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)


,rank,client_hash_id,content_hash_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_c182d11e4862a37d,content_a015022fdf20343b,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
1,2,client_c182d11e4862a37d,content_af7f2771f9386f92,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
2,3,client_c182d11e4862a37d,content_1003d0e15bd1a910,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
3,4,client_c182d11e4862a37d,content_686064cfde37e2b3,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
4,5,client_c182d11e4862a37d,content_455ee259df594322,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
5,6,client_c182d11e4862a37d,content_abaf64d1ed51568f,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
6,7,client_c182d11e4862a37d,content_69fa27a38394df56,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
7,8,client_c182d11e4862a37d,content_9f3f62757ed3dfa9,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
8,9,client_c182d11e4862a37d,content_b64e454944d3340f,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."
9,10,client_c182d11e4862a37d,content_5917bce4a37cd949,4,stale_low_visibility,review_refresh,Higher-priority baseline pick because the page...,"The page may be intentionally low-visibility, ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The weakest baseline picks are pages that receive a high score mainly because they have low observed search visibility. Low impressions do not necessarily mean that a page needs a refresh; they may reflect low search demand, intentional targeting of a small audience, seasonality, or a page that is not intended to compete in search.

I therefore treat the ranked queue as a review shortlist rather than an automatic action list.

### Leakage check

The baseline score uses only February 2026 information: content age calculated as of 2026-02-28 and February GSC impressions. It does not use March performance, the `went_dark` label, future-window fields, or product-generated priority/health/action flags.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — leakage check

print("Baseline inputs:")
print("- February 2026 content age")
print("- February 2026 GSC impressions")
print()
print("Excluded from the baseline:")
print("- March 2026 outcome / went_dark")
print("- Future-window performance")
print("- Product-generated flags or priority fields")

assert "went_dark" not in queue.columns
assert "health_score" not in queue.columns
assert "priority_score" not in queue.columns

print()
print("Leakage check: PASSED")


Baseline inputs:
- February 2026 content age
- February 2026 GSC impressions

Excluded from the baseline:
- March 2026 outcome / went_dark
- Future-window performance
- Product-generated flags or priority fields

Leakage check: PASSED


## Self-check

- ☑ Two signal checks are shown with visible bucket tables and n.
- ☑ At least one signal is linked to a real FlyRank flag.
- ☑ Both signals have clear verdicts.
- ☑ One baseline rule has a score, reason code, and action label.
- ☑ The ranked queue was written to `work/outputs/baseline_action_score.csv`.
- ☑ The top 20 have been reviewed with action, reason code, confidence note, and what would make each wrong.
- ☑ Weak picks and leakage risks are discussed.
- ☑ The leakage check passed.
- ☑ No future-window or label-derived inputs are used in the baseline.
- ☑ The notebook runs top to bottom with no errors.
- ☑ Claims use careful language: observed, measured, directional, decision-support.
- ☑ Committed under `work/notebooks/w04_baseline_score.ipynb`.